# U-RNN Quickstart — Urban Flood Nowcasting

<p align="center">
  <a href="https://doi.org/10.1016/j.jhydrol.2025.133117"><img src="https://img.shields.io/badge/Journal%20of%20Hydrology-2025-blue"/></a>
  <a href="https://github.com/holmescao/U-RNN"><img src="https://img.shields.io/github/stars/holmescao/U-RNN?style=social"/></a>
</p>

This notebook demonstrates the **U-RNN** model for high-resolution spatiotemporal urban flood nowcasting ([Cao et al., Journal of Hydrology 2025](https://doi.org/10.1016/j.jhydrol.2025.133117)).

## What this notebook covers

| Section | What it does | Time |
|---|---|---|
| **Part 1** | Setup — clone repo & install dependencies | ~15 s |
| **Part 2** | Quick demo — synthetic data, shows full pipeline | ~2 s |
| **Part 3** | Real inference — download UrbanFlood24 Lite weights + demo event, visualize flood maps | ~30 s |

> 💡 **To get real results fast:** Run Parts 1 + 3. For a GPU-free demo of the architecture, run Parts 1 + 2.

---

**Resources:**
- 📄 [Paper](https://doi.org/10.1016/j.jhydrol.2025.133117)
- 💾 [Dataset (UrbanFlood24)](https://holmescao.github.io/datasets/urbanflood24)
- 🤖 [GitHub](https://github.com/holmescao/U-RNN)
- ⚖️ [Pre-trained weights](https://github.com/holmescao/U-RNN#3-pre-trained-weights)

---
## Part 1 — Setup

Install PyTorch (CPU build for Colab free tier; GPU builds work too) and clone the repository.

In [ ]:
# Check GPU availability
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("GPU available:", result.stdout.strip())
    DEVICE = "0"
else:
    print("No GPU detected — running on CPU (inference only, Part 2 & 4 will be slow)")
    DEVICE = "cpu"

In [ ]:
import os

# Clone repo (skip if already present)
if not os.path.isdir('/content/U-RNN'):
    !git clone https://github.com/holmescao/U-RNN /content/U-RNN --quiet
else:
    print('Repo already cloned.')

%cd /content/U-RNN/code
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies (skip torch — already in Colab)
!pip install -q pyyaml tqdm openpyxl seaborn celluloid opencv-python-headless

# Verify
import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

---
## Part 2 — Architecture Demo (Synthetic Data)

This section creates a tiny **synthetic dataset** (random DEM + rainfall + flood) and runs one forward pass through the U-RNN model. No real data needed — this verifies that everything is installed correctly and shows the model input/output shapes.

In [ ]:
import torch
import numpy as np
import sys, os

sys.path.insert(0, '.')

# ── 1. Load model ─────────────────────────────────────────────────────────────
from src.lib.model.networks.model import ED
from src.lib.utils.net_config import load_net_config, get_input_channels
from src.lib.model.networks.net_params import get_network_params
from src.lib.utils.general import initialize_states

H, W = 64, 64          # tiny spatial grid for demo
T    = 12              # sequence length
historical_nums = 3    # look-back steps
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net_cfg = load_net_config()  # default network.yaml
input_channels = get_input_channels(net_cfg, historical_nums)
params = get_network_params(False, H, W, input_channels=input_channels, net_cfg=net_cfg)

model = ED(
    clstm_flag=False,           # use ConvGRU (not ConvLSTM)
    encoder_params=params[0],
    decoder_params=params[1],
    cls_thred=0.5,
    use_checkpoint=False,
    input_height=H,
    input_width=W,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded  |  Parameters: {total_params/1e6:.2f} M  |  Device: {device}')

# ── 2. Synthetic inputs ────────────────────────────────────────────────────────
# Each call: input shape (B, S, C, H, W) where S=1 for single-step inference
B = 1
C = input_channels

enc1, enc2, enc3, dec1, dec2, dec3 = initialize_states(
    device, H, W, net_cfg=net_cfg)

# ── 3. Forward pass (one timestep at a time) ───────────────────────────────────
model.eval()
outputs = []
with torch.no_grad():
    for t in range(T):
        x = torch.randn(B, 1, C, H, W, device=device)  # (B, S=1, C, H, W)
        out, enc1, enc2, enc3, dec1, dec2, dec3 = model(
            x, enc1, enc2, enc3, dec1, dec2, dec3)
        outputs.append(out.cpu())  # out: (B, 1, H, W)

outputs = torch.cat(outputs, dim=1)  # (B, T, H, W)
print(f'\nInput  shape: ({B}, 1, {C}, {H}, {W})  [batch, S=1, channels, height, width]')
print(f'Output shape: {tuple(outputs.shape)}  [batch, time_steps, height, width]')
print(f'Output range: [{outputs.min():.3f}, {outputs.max():.3f}]  (normalised flood depth)')
print('\n✅  Architecture demo complete — U-RNN forward pass works correctly.')

---
## Part 3 — Real Inference with Pre-trained Weights (UrbanFlood24 Lite, location1)

This section:
1. Downloads the **pre-trained UrbanFlood24 Lite weights** (location1, 128×128, R²=0.989) from Hugging Face
2. Downloads a **demo event** (r100y_p0.5_d3h, 1 test event, ~1.5 MB) for quick inference
3. Runs `test.py` and displays the **3-row comparison figure** (Reference / U-RNN / Error)

In [ ]:
import os, zipfile, glob

# ── Download pre-trained UrbanFlood24 Lite weights (location1) ─────────────────
WEIGHTS_HF_URL = 'https://huggingface.co/holmescao/U-RNN/resolve/main/checkpoints/loc1_lite_weights.zip'

# Check if any checkpoint already exists
existing = glob.glob('../exp/**/save_model/*.pth.tar', recursive=True)
if existing:
    ckpt_path = existing[0]
    print(f'Weights already present: {ckpt_path}')
else:
    print('Downloading location1 lite weights (~20 MB)...')
    !wget -q --show-progress -O /tmp/loc1_lite_weights.zip {WEIGHTS_HF_URL}

    with zipfile.ZipFile('/tmp/loc1_lite_weights.zip', 'r') as zf:
        names = zf.namelist()
        print(f'Zip contains {len(names)} file(s):')
        for n in names[:5]:
            print(f'  {n}')
        # Zip internal structure is  exp/<timestamp>/save_model/...
        # so extract one level up (..) to land in ../exp/<timestamp>/...
        zf.extractall('..')

    found = glob.glob('../exp/**/save_model/*.pth.tar', recursive=True)
    if not found:
        found = glob.glob('../exp/**/*.pth.tar', recursive=True)
    if not found:
        raise FileNotFoundError('No .pth.tar found after extraction — check zip structure above.')
    ckpt_path = found[0]
    print(f'Checkpoint found: {ckpt_path}')

# Extract TIMESTAMP robustly: path = ../exp/<TIMESTAMP>/save_model/checkpoint.pth.tar
parts = ckpt_path.replace('\\', '/').split('/')
try:
    exp_pos = next(i for i, p in enumerate(parts) if p == 'exp')
    TIMESTAMP = parts[exp_pos + 1]
except (StopIteration, IndexError):
    TIMESTAMP = '20260316_130418_443889'
    print(f'Warning: could not parse TIMESTAMP from path, using default.')

CKPT_NAME = os.path.basename(ckpt_path)
size_mb   = os.path.getsize(ckpt_path) / 1e6
print(f'File size: {size_mb:.1f} MB  |  TIMESTAMP: {TIMESTAMP}  |  CKPT: {CKPT_NAME}')

In [ ]:
import os, zipfile

# ── Download demo dataset (1 event from UrbanFlood24 Lite location1, ~1.5 MB) ──
# Contains: flood.npy (36, 128, 128) + rainfall.npy (36,) + geodata (DEM, impervious, manhole)

DEMO_DATA_HF_URL = 'https://huggingface.co/holmescao/U-RNN/resolve/main/demo_data/loc1_lite_demo_r100y.zip'
DATA_ROOT  = '../data/urbanflood24_lite'
DEMO_ZIP   = '/tmp/loc1_lite_demo_r100y.zip'
DEMO_EVENT = 'r100y_p0.5_d3h'
LOCATION   = 'location1'

demo_event_path = os.path.join(DATA_ROOT, 'test/flood', LOCATION, DEMO_EVENT)

if os.path.isdir(demo_event_path):
    print(f'Demo data already present: {demo_event_path}')
else:
    print('Downloading demo data (~1.5 MB)...')
    !wget -q --show-progress -O {DEMO_ZIP} {DEMO_DATA_HF_URL}
    print('Extracting...')
    with zipfile.ZipFile(DEMO_ZIP, 'r') as zf:
        zf.extractall('../data')
    print(f'Demo data extracted → {DATA_ROOT}')

# Verify
assert os.path.isfile(os.path.join(demo_event_path, 'flood.npy')),    'flood.npy not found!'
assert os.path.isfile(os.path.join(demo_event_path, 'rainfall.npy')), 'rainfall.npy not found!'
print(f'flood.npy + rainfall.npy verified in {demo_event_path}')

import numpy as np
flood_shape = np.load(os.path.join(demo_event_path, 'flood.npy')).shape
print(f'flood.npy shape: {flood_shape}  (T=36 steps, 128×128 grid)')

In [ ]:
import os

with open('/tmp/loc1_lite_demo.txt', 'w') as f:
    f.write(f'{DEMO_EVENT}\n')

print(f'Running inference (location1, {DEMO_EVENT})...')
!python test.py \
    --exp_config configs/lite.yaml \
    --timestamp  {TIMESTAMP} \
    --data_root  {DATA_ROOT} \
    --location   {LOCATION} \
    --test_list_file /tmp/loc1_lite_demo.txt \
    --device {DEVICE}

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pattern = f'../exp/{TIMESTAMP}/figs/**/{DEMO_EVENT}/water_depth_spatial_temporal.png'
figs = sorted(glob.glob(pattern, recursive=True))

if figs:
    img = mpimg.imread(figs[-1])
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'U-RNN — UrbanFlood24 Lite | location1 | {DEMO_EVENT}  (R²=0.989)\n'
        'Row 1: Reference (MIKE+)  |  Row 2: U-RNN prediction  |  Row 3: Absolute error',
        fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Figure: {figs[-1]}')
else:
    print('No figure found — check test.py output above for errors.')

---
## Part 4 — Training Demo (Synthetic Data, 5 Epochs)

This section generates a **tiny synthetic dataset** (8 training + 4 test events, 32×32×12) and runs **5 training epochs** to demonstrate the full training pipeline. Loss should decrease from random initialization.

> ⏱ Estimated time: ~1–2 minutes on GPU, ~3–5 minutes on CPU.

In [ ]:
import numpy as np
import os

# ── Generate a minimal synthetic dataset ──────────────────────────────────────
# Structure mirrors UrbanFlood24: train/test × flood/<location>/<event>/ + geodata

SYNTH_ROOT = '/tmp/urnn_synth'
H, W, T = 32, 32, 12
LOCATION = 'synth_loc'
TRAIN_EVENTS = [f'event_{i:02d}' for i in range(8)]
TEST_EVENTS  = [f'test_{i:02d}'  for i in range(4)]

np.random.seed(42)

for split, events in [('train', TRAIN_EVENTS), ('test', TEST_EVENTS)]:
    # Geodata
    geo_dir = os.path.join(SYNTH_ROOT, split, 'geodata', LOCATION)
    os.makedirs(geo_dir, exist_ok=True)
    np.save(os.path.join(geo_dir, 'absolute_DEM.npy'),  np.random.rand(H, W).astype('float32') * 10)
    np.save(os.path.join(geo_dir, 'impervious.npy'),    np.random.rand(H, W).astype('float32'))
    np.save(os.path.join(geo_dir, 'manhole.npy'),       (np.random.rand(H, W) > 0.95).astype('float32'))

    # Events
    for ev in events:
        ev_dir = os.path.join(SYNTH_ROOT, split, 'flood', LOCATION, ev)
        os.makedirs(ev_dir, exist_ok=True)
        # rainfall: scalar (T,)
        np.save(os.path.join(ev_dir, 'rainfall.npy'),  (np.random.rand(T) * 30).astype('float32'))
        # flood: (T, H, W) in metres, sparse (mostly dry)
        flood = np.random.rand(T, H, W).astype('float32') * 0.3
        flood[flood < 0.25] = 0.0   # most cells dry
        np.save(os.path.join(ev_dir, 'flood.npy'), flood)

# Event list files
TRAIN_TXT = '/tmp/synth_train.txt'
TEST_TXT  = '/tmp/synth_test.txt'
with open(TRAIN_TXT, 'w') as f: f.write('\n'.join(TRAIN_EVENTS) + '\n')
with open(TEST_TXT,  'w') as f: f.write('\n'.join(TEST_EVENTS)  + '\n')

print(f'✅  Synthetic dataset created at {SYNTH_ROOT}')
print(f'   Train: {len(TRAIN_EVENTS)} events  |  Test: {len(TEST_EVENTS)} events')
print(f'   Grid: {H}×{W}×{T} steps  (location: {LOCATION})')

In [ ]:
import subprocess, sys

# ── Run 5-epoch training on synthetic data ─────────────────────────────────────
cmd = [
    sys.executable, 'main.py',
    '--data_root',       SYNTH_ROOT,
    '--location',        LOCATION,
    '--train_list_file', TRAIN_TXT,
    '--test_list_file',  TEST_TXT,
    '--input_height',    str(H),
    '--input_width',     str(W),
    '--historical_nums', '2',
    '--duration',        str(T),
    '--window_size',     str(T),
    '--seq_num',         '4',
    '--epochs',          '5',
    '--batch_size',      '1',
    '--num_workers',     '0',
    '--flood_max',       '5000',
    '--rain_max',        '60.0',
    '--cumsum_rain_max', '250.0',
    '--flood_thres',     '150.0',
    '--viz_time_points', '0', '3', '7', '11',
    '--lr',              '0.01',
    '--lr_min',          '0.0001',
    '--schedule_name',   'WarmUpCosineAnneal',
    '--warm_up_iter',    '2',
    '--loss_name',       'FocalBCE_and_WMSE',
    '--reduction',       'mean',
    '--cls_thred',       '0.5',
    '--prewarming',      'false',
    '--use_checkpoint',
    '--device',          DEVICE,
    '--test',            'true',
]

print('Training command:', ' '.join(cmd[:6]), '...')
print('='*60)

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print('\n✅  Training completed successfully!')
else:
    print('\n❌  Training returned non-zero exit code.')

In [ ]:
import glob, os, re
import matplotlib.pyplot as plt

# ── Parse and plot training loss ───────────────────────────────────────────────
# Find the most recent experiment log
log_files = sorted(glob.glob('../exp/*/timestamp.txt'), key=os.path.getmtime)

if log_files:
    exp_dir = os.path.dirname(log_files[-1])
    log_file = glob.glob(os.path.join(exp_dir, '*.log'))
    if log_file:
        with open(log_file[0]) as f:
            lines = f.readlines()

        epochs, losses = [], []
        for line in lines:
            m = re.search(r'\[\s*(\d+)/\d+\].*loss:(\d+\.\d+)', line)
            if m:
                epochs.append(int(m.group(1)))
                losses.append(float(m.group(2)))

        if epochs:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(epochs, losses, 'b-o', markersize=5, linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Training Loss')
            ax.set_title('U-RNN Training Loss — Synthetic Data (5 Epochs)')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            print(f'Loss at epoch {epochs[0]}: {losses[0]:.4f}  →  epoch {epochs[-1]}: {losses[-1]:.4f}')
        else:
            print('Could not parse loss values from log file.')
    else:
        print('Log file not found.')
else:
    print('No experiment directory found.')

---
## Next Steps

🎉 You've completed the U-RNN quickstart!

**Recommended progression (from the [README](https://github.com/holmescao/U-RNN)):**

1. **Scenario B — Lite training (~30 min):** Download the [UrbanFlood24 Lite dataset](https://github.com/holmescao/U-RNN#2-dataset-preparation) (~few hundred MB) and train:
   ```bash
   python main.py --exp_config configs/lite.yaml
   ```
2. **Scenario C — LarNO datasets:** Train on [Futian or UKEA](https://holmescao.github.io/datasets/LarNO) after converting with `tools/convert_larfno_data.py`.
3. **Scenario D — Full paper results (~40 h):** Download the [full UrbanFlood24 dataset](https://holmescao.github.io/datasets/urbanflood24) (~115 GB) and train:
   ```bash
   python main.py --exp_config configs/location1_scratch.yaml
   ```

**To use your own data:**
- Prepare `flood.npy` (T, H, W), `rainfall.npy` (T,) or (T, H, W), and geodata files
- Copy `configs/lite.yaml` to `configs/custom.yaml` and update `data_root`, `input_height`, `input_width`
- Run `python main.py --exp_config configs/custom.yaml`

**Citation:**
```bibtex
@article{cao2025u,
  title={U-RNN high-resolution spatiotemporal nowcasting of urban flooding},
  author={Cao, Xiaoyan and Wang, Baoying and Yao, Yao and Zhang, Lin and Xing, Yanwen
          and Mao, Junqi and Zhang, Runqiao and Fu, Guangtao
          and Borthwick, Alistair GL and Qin, Huapeng},
  journal={Journal of Hydrology},
  pages={133117},
  year={2025},
  publisher={Elsevier}
}
```